# 02 - Main run

Runs the four methods through the curriculum on a real LoRA. Written to be run
repeatedly: each session does as many nights as it has time for, checkpoints
them, and picks up from the last one next time.

Set `NIGHTS_THIS_SESSION` to whatever the session budget allows.

## Environment

In [ ]:
import os, sys, subprocess

REPO = "/kaggle/working/skill-library"
if not os.path.exists(REPO):
    # On Kaggle, add the repo as a dataset or clone it here.
    REPO = os.getcwd()
sys.path.insert(0, REPO)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "peft>=0.11", "transformers>=4.40", "accelerate"], check=False)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{name}  {total:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")

## Configuration

In [ ]:
from baselines.gpu_runner import GPURun, load_config
from sleep.lora import load_base_model

config = load_config("configs/default.yaml")
config["run"]["run_id"] = "gpu_main"

# One night budget per session. Raise it if the session has time left;
# whatever finishes is checkpointed, so nothing is lost either way.
NIGHTS_THIS_SESSION = 2
METHODS = ["ours", "seq_no_replay", "isolated", "in_context"]

model, tokenizer = load_base_model(
    config["model"]["name"], config["model"]["device"], config["model"]["dtype"]
)

## Run

Each method gets its own checkpoint directory, so they can be run in any order
and across any number of sessions. A method that has already finished is
skipped rather than re-run.

In [ ]:
import time, torch

status = {}
for method in METHODS:
    run = GPURun(config, method=method, model=model, tokenizer=tokenizer)
    resumed = run.resume()
    if resumed:
        print(f"[{method}] resuming from night {resumed}, {len(run.library)} skills")
    if len(run.library) == 15 and not run.consolidator.pending:
        print(f"[{method}] already complete, skipping")
        status[method] = "complete"
        continue

    started = time.time()
    run.run(max_nights=NIGHTS_THIS_SESSION)
    status[method] = (
        f"night {run.consolidator.night_count}, {len(run.library)}/15 skills, "
        f"{(time.time()-started)/60:.1f} min"
    )
    print(f"[{method}] {status[method]}")

    del run
    torch.cuda.empty_cache()

status

## Progress

In [ ]:
# Where each method stands. Re-run this notebook until every method reaches 15.
from pathlib import Path
from sleep.checkpoint import latest_night

root = Path(config["run"]["root"]) / config["run"]["run_id"]
for method in METHODS:
    night = latest_night(root / method)
    print(f"{method:<16} last night: {night}")

## Collect results

Only meaningful once every method has reached 15 skills.

In [ ]:
from analysis.collect import summary_table, write_collected

path = write_collected(root, METHODS, run_id=config["run"]["run_id"])
print(path)

import json
print(summary_table(json.loads(path.read_text())))